# Evaluación del asistente

Proceso **separado** del asistente. Equivale a correr en terminal:

```bash
python -m evaluacion.run --aa
```

Compara `v1_articulo` (un fragmento por artículo) contra `v2_fraccion` (un fragmento por fracción), y repite v1 sin caché para estimar el ruido entre corridas. Las llamadas al modelo se guardan en caché: re-ejecutar es gratis y da exactamente los mismos números.

In [ ]:
import os, sys
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ)); os.chdir(RAIZ)

import laboral.config
if not os.environ.get("ANTHROPIC_API_KEY"):
    from getpass import getpass
    os.environ["ANTHROPIC_API_KEY"] = getpass("ANTHROPIC_API_KEY: ")

In [ ]:
from evaluacion.run import main
df = main(["--aa"])

## Reporte

In [ ]:
from IPython.display import Markdown, display
display(Markdown((RAIZ / "resultados" / "reporte.md").read_text(encoding="utf-8").replace("](", "](resultados/")))

## Explorar por pregunta

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 120)
cols = ["id", "version", "tipo", "estado", "acierto_k", "fidelidad", "cobertura", "contradice", "exito", "error_principal", "costo_usd", "segundos_total"]
df[cols].sort_values(["id", "version"])

In [ ]:
piv = df.pivot_table(index="id", columns="version", values="exito")
piv[piv.nunique(axis=1) > 1]

In [ ]:
ID = "P31"
for _, f in df[df.id == ID].iterrows():
    print(f"[{f.version}] {f.estado} | éxito={f.exito} | {f.error_principal}\n{f.respuesta}\n")